# Netflix Tavsiye Sistemi

In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv('combined_data_1.txt', header=None, names=['Cust_Id', 'Rating'], usecols=[0,1])
df['Rating'] = df['Rating'].astype(float)

In [3]:
#FİLM ID'LERİNİN AYRI BİR SÜTUNA ÇIKARILMASI
#Makaledeki mantıkla: Rating sütunu boş olan satırlar aslında Film ID'sini belirtir (Örn: 1:, 2:)
df_nan = pd.DataFrame(pd.isnull(df.Rating))
df_nan = df_nan[df_nan['Rating'] == True]
df_nan = df_nan.reset_index()

movie_np = []
movie_id = 1

In [4]:
# Film ID'lerinin hangi satır aralıklarında olduğunu hesaplayıp listeye ekliyoruz
for i, j in zip(df_nan['index'][1:], df_nan['index'][:-1]):
    temp = np.full((1, i - j - 1), movie_id)
    movie_np.extend(temp[0])
    movie_id += 1

In [5]:
# Son film grubu için ekleme yapıyoruz
last_record = np.full((1, len(df) - df_nan['index'].iloc[-1] - 1), movie_id)
movie_np.extend(last_record[0])

In [6]:
# Boş satırları (Film ID satırlarını) ana tablodan temizliyoruz
df = df[pd.notnull(df['Rating'])]
df['Movie_Id'] = movie_np
df['Cust_Id'] = df['Cust_Id'].astype(int)

In [7]:
df.head()

,Cust_Id,Rating,Movie_Id
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1


In [8]:
# Sistem hafızasının (RAM) çökmemesi için az puan veren kullanıcıları ve az puan alan filmleri eliyoruz
min_movie_ratings = 5000  # En az 5000 puan almış popüler filmler
min_user_ratings = 50     # En az 50 filme puan vermiş aktif kullanıcılar

In [9]:
filter_movies = df['Movie_Id'].value_counts() > min_movie_ratings
filter_movies = filter_movies[filter_movies].index

In [10]:
filter_users = df['Cust_Id'].value_counts() > min_user_ratings
filter_users = filter_users[filter_users].index

In [11]:
df_filtered = df[df['Movie_Id'].isin(filter_movies) & df['Cust_Id'].isin(filter_users)]
print(f"Filtreleme sonrası satır sayısı: {len(df_filtered)}")

Filtreleme sonrası satır sayısı: 15970085


In [12]:
# Film isimlerini içeren dosyayı okuyoruz (Hatalı satırları atlamak için encoding ve error_bad_lines ayarlı)
df_title = pd.read_csv('movie_titles.csv', encoding="ISO-8859-1", header=None, names=['Movie_Id', 'Year', 'Name'], on_bad_lines='skip')
df_title.set_index('Movie_Id', inplace=True)

In [13]:
# Sütunlarda film isimlerinin görünmesi için Id'leri isimlerle değiştiriyoruz
df_filtered = df_filtered.merge(df_title['Name'], left_on='Movie_Id', right_index=True)


In [14]:
# Matrisi oluşturuyoruz (Satırlar: Kullanıcılar, Sütunlar: Filmler)
movie_matrix = df_filtered.pivot_table(index='Cust_Id', columns='Name', values='Rating')
print("Kullanıcı-Film matrisi başarıyla oluşturuldu.")

Kullanıcı-Film matrisi başarıyla oluşturuldu.


In [ ]:
print("Pearson korelasyon matrisi hesaplanıyor (Bu işlem birkaç dakika sürebilir)...")
corr_matrix = movie_matrix.corr(method='pearson')

Pearson korelasyon matrisi hesaplanıyor (Bu işlem birkaç dakika sürebilir)...


In [ ]:
joblib.dump(corr_matrix, "netflix_real_corr.pkl")

In [ ]:
# Aktif olan film listesini de arayüzde göstermek için kaydediyoruz
active_movies = pd.DataFrame(df_filtered['Name'].unique(), columns=['Movie_Title'])
active_movies.to_csv("active_movies.csv", index=False)